In [ ]:
import duckdb
import pandas as pd
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
import warnings

warnings.filterwarnings('ignore')


con = duckdb.connect()

ratings_path = Path('..', 'data', 'raw', 'ml-25m', 'ratings.csv')
movies_path = Path('..', 'data', 'raw', 'ml-25m', 'movies.csv')

print("\nЗагрузка данных")

ratings_df = pd.read_csv(ratings_path)
movies_df = pd.read_csv(movies_path)

print(f"raw_ratings: {len(ratings_df):,} строк")
print(f"raw_movies: {len(movies_df):,} строк")

con.register('ratings_temp', ratings_df)
con.register('movies_temp', movies_df)
con.execute("CREATE TABLE raw_ratings AS SELECT * FROM ratings_temp")
con.execute("CREATE TABLE raw_movies AS SELECT * FROM movies_temp")

# Трансформация
con.execute("""
    CREATE TABLE user_activity AS
    SELECT 
        r.userId, r.movieId, r.rating, r.timestamp,
        to_timestamp(r.timestamp) AS datetime,
        CAST(to_timestamp(r.timestamp) AS DATE) AS date,
        DATE_PART('hour', to_timestamp(r.timestamp)) AS hour,
        m.title, m.genres
    FROM raw_ratings r
    LEFT JOIN raw_movies m ON r.movieId = m.movieId
""")

print(f"user_activity: {con.execute('SELECT COUNT(*) FROM user_activity').fetchone()[0]:,} строк")



# ## График 1: Активность пользователей по дням недели

dow_activity = con.execute("""
    SELECT 
        CASE 
            WHEN DATE_PART('dow', date) = 0 THEN 6
            ELSE DATE_PART('dow', date) - 1
        END as day_of_week_monday,
        COUNT(DISTINCT userId) as dau
    FROM user_activity
    GROUP BY day_of_week_monday
    ORDER BY day_of_week_monday
""").df()

days = ['Пн', 'Вт', 'Ср', 'Чт', 'Пт', 'Сб', 'Вс']
dow_activity['day_name'] = dow_activity['day_of_week_monday'].astype(int).map(lambda x: days[x])

fig = px.bar(
    dow_activity,
    x='day_name',
    y='dau',
    title='Активность пользователей по дням недели',
    labels={'day_name': 'День недели', 'dau': 'Уникальные пользователи (DAU)'},
    color='dau',
    color_continuous_scale='Blues',
    text_auto=True
)

fig.update_layout(
    width=1000, height=550,
    margin=dict(l=60, r=60, t=60, b=60),
    showlegend=False,
    xaxis={'categoryorder': 'array', 'categoryarray': ['Пн', 'Вт', 'Ср', 'Чт', 'Пт', 'Сб', 'Вс']},
    font=dict(size=13)
)

fig.show()

# ## График 2: Распределение оценок

rating_dist = con.execute("""
    SELECT 
        rating,
        COUNT(*) as count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
    FROM raw_ratings
    GROUP BY rating
    ORDER BY rating
""").df()

fig = go.Figure(data=[
    go.Bar(
        x=rating_dist['rating'],
        y=rating_dist['count'],
        marker_color='gold',
        marker_line_color='darkorange',
        marker_line_width=2,
        text=rating_dist['percentage'].astype(str) + '%',
        textposition='auto'
    )
])

fig.update_layout(
    title='Распределение оценок пользователей',
    xaxis_title='Оценка',
    yaxis_title='Количество',
    xaxis=dict(tickmode='linear', tick0=1, dtick=1),
    width=1000, height=550,
    margin=dict(l=60, r=60, t=60, b=60),
    showlegend=False,
    font=dict(size=13)
)

fig.show()


# ## График 3: Топ-20 жанров по популярности

genre_stats = con.execute("""
    WITH genre_expanded AS (
        SELECT 
            userId, movieId, rating,
            TRIM(UNNEST(STRING_SPLIT(genres, '|'))) as genre
        FROM user_activity
        WHERE genres IS NOT NULL AND genres != '(no genres listed)'
    )
    SELECT 
        genre,
        COUNT(DISTINCT userId) as unique_users,
        COUNT(*) as total_views,
        ROUND(AVG(rating), 2) as avg_rating
    FROM genre_expanded
    GROUP BY genre
    ORDER BY total_views DESC
    LIMIT 20
""").df()

fig = px.bar(
    genre_stats,
    x='total_views',
    y='genre',
    orientation='h',
    title='Топ-20 жанров по популярности',
    labels={'total_views': 'Количество просмотров', 'genre': 'Жанр'},
    color='total_views',
    color_continuous_scale='Oranges',
    text_auto=True
)

fig.update_layout(
    width=1100, height=800,
    margin=dict(l=120, r=60, t=60, b=60),
    yaxis={'categoryorder': 'total ascending'},
    showlegend=False,
    font=dict(size=13)
)

fig.show()


# ## График 4: Коэффициент удержания (Retention)

retention = con.execute("""
    WITH user_first_day AS (
        SELECT userId, MIN(date) as first_day
        FROM user_activity
        GROUP BY userId
    ),
    user_next_day AS (
        SELECT 
            ufd.userId, ufd.first_day,
            MAX(CASE WHEN ua.date = ufd.first_day + INTERVAL '1 day' THEN 1 ELSE 0 END) as returned_next_day
        FROM user_first_day ufd
        LEFT JOIN user_activity ua ON ufd.userId = ua.userId 
        GROUP BY ufd.userId, ufd.first_day
    )
    SELECT 
        COUNT(DISTINCT userId) as cohort_size,
        SUM(returned_next_day) as returned,
        ROUND(SUM(returned_next_day) * 100.0 / COUNT(DISTINCT userId), 2) as retention_rate
    FROM user_next_day
""").fetchall()

cohort_size = retention[0][0]
returned = retention[0][1]
not_returned = cohort_size - returned

fig = go.Figure(data=[go.Pie(
    labels=['Вернулись на следующий день', 'Не вернулись'],
    values=[returned, not_returned],
    hole=.4,
    marker_colors=['#4CAF50', '#F44336'],
    textinfo='label+percent+value',
    textposition='outside',
    automargin=True,
    textfont=dict(size=13)
)])

fig.update_layout(
    title='Retention Day 1 (коэффициент удержания)',
    width=1000, height=600,
    margin=dict(l=60, r=60, t=60, b=60),
    showlegend=True,
    legend=dict(font=dict(size=13))
)

fig.show()

print(f"\n Retention Rate: {retention[0][2]:.1f}%")
print(f"   Когорта: {cohort_size:,} пользователей")
print(f"   Вернулись: {returned:,} пользователей")


# ## График 5: Активность по часам суток

hourly_activity = con.execute("""
    SELECT 
        hour,
        COUNT(DISTINCT userId) as dau,
        COUNT(*) as total_ratings
    FROM user_activity
    GROUP BY hour
    ORDER BY hour
""").df()

fig = px.line(
    hourly_activity,
    x='hour',
    y='dau',
    title='Активность пользователей по часам суток',
    labels={'hour': 'Час суток', 'dau': 'Уникальные пользователи (DAU)'},
    markers=True,
    line_shape='spline'
)

fig.update_traces(
    line=dict(color='blue', width=3),
    marker=dict(size=8, color='blue')
)

fig.update_layout(
    width=1000, height=550,
    margin=dict(l=60, r=60, t=60, b=60),
    xaxis=dict(tickmode='linear', tick0=0, dtick=2),
    showlegend=False,
    font=dict(size=13),
    hovermode='x unified'
)

fig.show()

peak_hour = hourly_activity.loc[hourly_activity['dau'].idxmax()]
print(f"\n Пик активности: {int(peak_hour['hour'])}:00 ({peak_hour['dau']:,} пользователей)")


# ## График 6: Средний рейтинг по жанрам 

genre_rating_dist = con.execute("""
    WITH genre_expanded AS (
        SELECT 
            rating,
            TRIM(UNNEST(STRING_SPLIT(genres, '|'))) as genre
        FROM user_activity
        WHERE genres IS NOT NULL AND genres != '(no genres listed)'
    )
    SELECT 
        genre,
        ROUND(AVG(rating), 2) as avg_rating,
        COUNT(*) as count
    FROM genre_expanded
    GROUP BY genre
    HAVING COUNT(*) > 5000
    ORDER BY avg_rating DESC
""").df()

fig = px.bar(
    genre_rating_dist,
    x='genre',
    y='avg_rating',
    title='Средний рейтинг по жанрам',
    labels={'genre': 'Жанр', 'avg_rating': 'Средний рейтинг'},
    color='avg_rating',
    color_continuous_scale='RdYlGn',
    text_auto='.2f'
)

fig.update_layout(
    width=1000, height=650,
    margin=dict(l=60, r=60, t=60, b=80),
    xaxis={'tickangle': 45},
    showlegend=False,
    font=dict(size=13)
)

fig.show()

print(" Итоговая статистика")

total_users = con.execute("SELECT COUNT(DISTINCT userId) FROM raw_ratings").fetchone()[0]
total_movies = con.execute("SELECT COUNT(DISTINCT movieId) FROM raw_ratings").fetchone()[0]
total_ratings = con.execute("SELECT COUNT(*) FROM raw_ratings").fetchone()[0]
avg_rating = con.execute("SELECT ROUND(AVG(rating), 2) FROM raw_ratings").fetchone()[0]

print(f"\n Объём данных:")
print(f"   • Всего пользователей: {total_users:,}")
print(f"   • Всего фильмов: {total_movies:,}")
print(f"   • Всего оценок: {total_ratings:,}")
print(f"   • Средний рейтинг:  {avg_rating}")


con.close()


Загрузка данных
raw_ratings: 25,000,095 строк
raw_movies: 62,423 строк
user_activity: 25,000,095 строк



 Retention Rate: 18.8%
   Когорта: 162,541 пользователей
   Вернулись: 30,545 пользователей



 Пик активности: 23:00 (34,670 пользователей)


 Итоговая статистика

 Объём данных:
   • Всего пользователей: 162,541
   • Всего фильмов: 59,047
   • Всего оценок: 25,000,095
   • Средний рейтинг:  3.53
